# Model Training Stage — Multi-Label Tag Prediction

This notebook trains the baseline machine learning model for predicting
StackOverflow tags from question text. The dataset used here is the
model-ready dataset produced in the previous pipeline stage.

**Input**
- model_ready.parquet

**Output**
- tag_prediction_model_tfidf.pkl
- tfidf_vectorizer.pkl
- mlb_encoder.pkl
- tag_prediction_model_sbert.pkl
- modeling.parquet

## Load Model-Ready Dataset

Load the cleaned and filtered dataset prepared in the feature-engineering
pipeline stage. This dataset contains cleaned text and finalized label lists.

In [2]:
import pandas as pd

df = pd.read_parquet("model_ready.parquet")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset shape: (48911, 3)
Columns: ['id', 'clean_text', 'filtered_tags']


,id,clean_text,filtered_tags
0,12421444,how to format a number 0..9 to display with 2 ...,[java]
1,12468823,python datetime - setting fixed hour and minut...,[python]
2,12553160,getting visitors country from their ip i want ...,[php]
3,12583638,when is the @jsonproperty property used and wh...,[java]
4,12567578,what does the layoutinflater attachtoroot para...,[android]


## Multi-Label Encoding

StackOverflow questions may contain multiple tags.

Labels are transformed into a multi-label binary format using
MultiLabelBinarizer to enable supervised learning.

In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["filtered_tags"])

print(f"Label matrix shape: {Y.shape}")
print(f"Number of unique tags: {len(mlb.classes_)}")
print(f"Tags: {list(mlb.classes_)}")

Label matrix shape: (48911, 50)
Number of unique tags: 50
Tags: ['.net', 'algorithm', 'android', 'arrays', 'asp.net', 'asp.net-mvc', 'bash', 'c', 'c#', 'c++', 'c++11', 'css', 'database', 'django', 'eclipse', 'git', 'haskell', 'html', 'ios', 'iphone', 'java', 'javascript', 'jquery', 'json', 'linux', 'macos', 'multithreading', 'mysql', 'node.js', 'objective-c', 'performance', 'php', 'postgresql', 'python', 'r', 'regex', 'ruby', 'ruby-on-rails', 'ruby-on-rails-3', 'scala', 'spring', 'sql', 'sql-server', 'string', 'visual-studio', 'visual-studio-2010', 'windows', 'wpf', 'xcode', 'xml']


## Text Feature Extraction (TF-IDF)

Question text is transformed into numerical features using TF-IDF
vectorization to represent term importance across the corpus.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    stop_words="english"
)

X = vectorizer.fit_transform(df["clean_text"])

print(f"TF-IDF matrix shape: {X.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

TF-IDF matrix shape: (48911, 50000)
Vocabulary size: 50000


## Train-Test Split

The dataset is divided into training and testing subsets to evaluate
model generalization performance.

> **Note:** `random_state=42` is fixed here and must be reused identically
> in the SBERT split and evaluation notebook to ensure both models
> are tested on the same samples.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size:  {X_test.shape[0]} samples")

Train size: 39128 samples
Test size:  9783 samples


## Train Baseline Classification Model (TF-IDF + LinearSVC)

A OneVsRest LinearSVC classifier is trained on TF-IDF features.
This serves as the baseline model for multi-label tag prediction.

In [6]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(LinearSVC(max_iter=2000), n_jobs=-1)
model.fit(X_train, Y_train)

print("TF-IDF model training complete.")

TF-IDF model training complete.


## Sample Prediction Check (TF-IDF Model)

A quick sanity check to verify the trained model produces
reasonable tag predictions before saving artifacts.

In [7]:
sample_preds = model.predict(X_test[:5])
decoded = mlb.inverse_transform(sample_preds)

for i, tags in enumerate(decoded):
    print(f"Sample {i}: {tags}")

Sample 0: ('python',)
Sample 1: ('java', 'scala')
Sample 2: ('java',)
Sample 3: ('c#',)
Sample 4: ('html',)


## Save TF-IDF Model Artifacts

The trained TF-IDF model, vectorizer, and label encoder are saved
for reuse in the evaluation and deployment stages.

In [8]:
import joblib

# Versioned copies for explicit pipeline tracking
joblib.dump(mlb,        "mlb_encoder_tfidf.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(model,      "tag_prediction_model_tfidf.pkl")

print("TF-IDF artifacts saved:")
print("  mlb_encoder.pkl")
print("  tfidf_vectorizer.pkl")
print("  tag_prediction_model.pkl")
print("  tag_prediction_model_tfidf.pkl")

TF-IDF artifacts saved:
  mlb_encoder.pkl
  tfidf_vectorizer.pkl
  tag_prediction_model.pkl
  tag_prediction_model_tfidf.pkl


---
## Sentence-BERT Embeddings (GPU-Accelerated)

Sentence-BERT generates dense semantic embeddings using a pre-trained
transformer. Unlike TF-IDF, these capture meaning beyond keyword overlap.

**GPU Setup:**
- Device is auto-detected: CUDA GPU if available, else CPU
- `batch_size=64` is used for GPU throughput (safe for most 4GB+ VRAM cards)
- The encoder is loaded directly from HuggingFace — **not** saved via joblib,
  as SentenceTransformer objects do not reliably restore GPU state through pickle

> **Why not joblib for SBERT?**  
> `SentenceTransformer` wraps PyTorch modules. Joblib-pickling a GPU model
> can silently restore it on CPU or fail entirely depending on the environment.
> The correct pattern is to always reload from the model ID — it takes seconds
> and is guaranteed to be correct.

In [9]:
import torch
from sentence_transformers import SentenceTransformer

# Auto-detect GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU found — running on CPU. Embedding generation will be slow.")

# Load SBERT model onto detected device
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"\nSentence-BERT loaded: all-MiniLM-L6-v2 on {device}")

Using device: cuda
GPU:  Tesla T4
VRAM: 15.64 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Sentence-BERT loaded: all-MiniLM-L6-v2 on cuda


In [10]:
# Generate embeddings for the full dataset
# batch_size=64 is optimal for GPU; reduce to 32 if you hit VRAM OOM errors
print("Generating Sentence-BERT embeddings...")

sbert_embeddings = sbert_model.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True,
    device=device
)

print(f"Embedding shape: {sbert_embeddings.shape}")
print(f"Dtype: {sbert_embeddings.dtype}")

Generating Sentence-BERT embeddings...


Batches:   0%|          | 0/765 [00:00<?, ?it/s]

Embedding shape: (48911, 384)
Dtype: float32


## Train-Test Split for SBERT

The same `random_state=42` and `test_size=0.2` as the TF-IDF split.
This is mandatory — mismatched splits make model comparison invalid.

In [11]:
sbert_train, sbert_test, Y_train_sbert, Y_test_sbert = train_test_split(
    sbert_embeddings, Y, test_size=0.2, random_state=42
)

print(f"SBERT train: {sbert_train.shape}")
print(f"SBERT test:  {sbert_test.shape}")

SBERT train: (39128, 384)
SBERT test:  (9783, 384)


## Train SBERT + LinearSVC Model

The same OneVsRest LinearSVC classifier is trained on SBERT embeddings.
This isolates the effect of the feature representation — the classifier
is identical, only the input features differ.

In [12]:
print("Training Sentence-BERT + LinearSVC model...")

sbert_clf = OneVsRestClassifier(LinearSVC(max_iter=2000), n_jobs=-1)
sbert_clf.fit(sbert_train, Y_train_sbert)

print("SBERT model training complete.")

Training Sentence-BERT + LinearSVC model...
SBERT model training complete.


## Sample Prediction Check (SBERT Model)

Quick sanity check on the SBERT model before saving.

In [13]:
sample_sbert_preds = sbert_clf.predict(sbert_test[:5])
decoded_sbert = mlb.inverse_transform(sample_sbert_preds)

for i, tags in enumerate(decoded_sbert):
    print(f"Sample {i}: {tags}")

Sample 0: ('python',)
Sample 1: ('java', 'scala')
Sample 2: ('java',)
Sample 3: ()
Sample 4: ('html',)


## Save SBERT Model Artifacts

Only the trained LinearSVC classifier is saved via joblib — it is a
standard sklearn object and serializes correctly.

The `SentenceTransformer` encoder is **not** saved via joblib. It must
be reloaded from the model ID in the evaluation notebook.

In [14]:
joblib.dump(sbert_clf, "tag_prediction_model_sbert.pkl")

print("SBERT artifacts saved:")
print("  tag_prediction_model_sbert.pkl  (LinearSVC classifier)")
print("")
print("NOTE: SentenceTransformer encoder is NOT saved via joblib.")
print("      Reload with: SentenceTransformer('all-MiniLM-L6-v2', device=device)")

SBERT artifacts saved:
  tag_prediction_model_sbert.pkl  (LinearSVC classifier)

NOTE: SentenceTransformer encoder is NOT saved via joblib.
      Reload with: SentenceTransformer('all-MiniLM-L6-v2', device=device)


## Export Processed Dataset

The processed dataset is saved in parquet format to reduce storage size
and speed up downstream ML pipeline loading.

In [15]:
df.to_parquet('modeling.parquet')
print("Dataset saved: modeling.parquet")
print(f"Shape: {df.shape}")

Dataset saved: modeling.parquet
Shape: (48911, 3)


## Summary

| Artifact | Description |
|---|---|
| `mlb_encoder.pkl` | MultiLabelBinarizer fitted on top-50 tags |
| `tfidf_vectorizer.pkl` | TF-IDF vectorizer (50k features, bigrams) |
| `tag_prediction_model.pkl` | TF-IDF + LinearSVC classifier |
| `tag_prediction_model_tfidf.pkl` | Same as above (versioned copy) |
| `tag_prediction_model_sbert.pkl` | SBERT + LinearSVC classifier |
| `modeling.parquet` | Full processed dataset for evaluation stage |

**Key decisions:**
- `max_iter=2000` on LinearSVC prevents convergence warnings on larger label spaces
- `n_jobs=-1` uses all CPU cores for OneVsRest parallel training
- `batch_size=64` for SBERT encoding — reduce to 32 if VRAM is limited
- SBERT encoder reloaded from HuggingFace in evaluation (not joblib) for reliable GPU restoration
- Both models use identical `random_state=42` / `test_size=0.2` split for valid comparison